# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
# YOUR CODE HERE

import pandas as pd
import numpy as np
import torch
import json
import os

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
# YOUR CODE HERE

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded successfully!")


W1121 14:17:10.963000 31940 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1121 14:17:11.322000 31940 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


In [3]:

query = "i need information about delivery periods"

messages = [
    {"role": "user", "content": query}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Customer Query:", query)
print("\nBaseline Model Response:", response)


Customer Query: i need information about delivery periods

Baseline Model Response: The delivery period for an item typically refers to the time frame during which it is expected to be delivered from the point of purchase until its arrival at the customer's location. This can vary depending on several factors including shipping speed, distance, and any additional services or options that may be included in the order.

Here’s how you might consider different aspects of delivery periods:

1. **Shipping Speed**: Different countries have varying shipping speeds. For example:
   - In the United States, items often arrive within 3-5 days.
   - In Europe, deliveries can take up to 7-1


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [4]:
import json
import re

# read the baseline answer we saved earlier
with open("outputs.json", "r", encoding="utf-8") as f:
    saved = json.load(f)

baseline_answer = saved["baseline_output"]
correct_rule = saved["ground_truth"]

print("Customer question:", saved["test_query"])
print("Correct rule:", correct_rule)
print()

text = baseline_answer.lower()

# check 1: does the answer give the real delivery time?
says_3_to_7 = "3-7" in text or "3–7" in text
print("Mentions 3-7 business days:", says_3_to_7)

# check 2: does it quote delivery times that are not in the company SOP?
# Pick out every "number-number days" style range in the answer.
ranges = re.findall(r"\d+\s*[-–]\s*\d*\s*days?", text)
print("Delivery-time ranges quoted:", ranges)

# check 3: does it talk about specific places or companies from its own memory?
outside_words = ["united states", "europe", "ups", "fedex", "dhl", "usps"]
for word in outside_words:
    print("Mentions", word, ":", word in text)

# check 4: was the answer cut off?
print("Cut off mid-sentence:", not baseline_answer.strip().endswith((".", "!", "?")))

Customer question: i need information about delivery periods
Correct rule: Domestic orders deliver within 3-7 business days.

Mentions 3-7 business days: False
Delivery-time ranges quoted: ['3-5 days']
Mentions united states : True
Mentions europe : True
Mentions ups : False
Mentions fedex : False
Mentions dhl : False
Mentions usps : False
Cut off mid-sentence: True


##### 3.1.2 Evaluate Baseline Quality Conclusion

**Test query:** "i need information about delivery periods"
**Correct rule (SOP):** "Domestic orders deliver within 3–7 business days."

**1. Comparison against the known rule**

The baseline model did not mention the 3–7 business day timeline from the SOP. It explained in general terms what a delivery period is and said it depends on shipping speed, distance and extra services.

Verdict: the answer is factually incomplete, because the one fact the customer needed is missing.

**2. Invented timelines, tracking systems or departments?**

- **Invented timelines: yes.** The model quoted delivery times from its own memory: "In the United States, items often arrive within 3-5 days" and "In Europe, deliveries can take up to 7-1…" (cut off). These are not the company's figures. The US range also does not match the SOP's 3–7 business days.
- No non-existent department was mentioned in the visible text.

**3. Unsupported statements (hallucinations)**

Specific claims with no support in the provided documents:
- Items in the United States "often arrive within 3-5 days".
- Deliveries in Europe "can take up to 7-1…" days.

Generic statements that are plausible but unverified:
- The delivery period depends on shipping speed, distance and extra services.

**4. Other quality issue**

The response was cut off in the middle of a number ("7-1") at the 120 token limit, so it is also incomplete in format.

**Learner Inference:**

The baseline produces a fluent, reasonable-sounding answer but does not give business-specific information. It missed the actual SOP rule (3–7 business days) and replaced it with general delivery times for specific countries that the knowledge base does not mention.

This justifies Stage 3, where retrieval supplies the relevant SOP before the model answers. It also justifies Stage 4, where a fine-tuned model acts as an intent router: it extracts a structured JSON intent from a messy customer query, and that intent is used to retrieve the correct SOP. Together, retrieval and routing aim to make answers more accurate and consistent.

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [5]:
# YOUR CODE HERE

ground_truth = "Domestic orders deliver within 3-7 business days."

baseline_data = {
    "test_query": query,
    "ground_truth": ground_truth,
    "baseline_output": response
}

with open("outputs.json", "w", encoding="utf-8") as f:
    json.dump(baseline_data, f, indent=4, ensure_ascii=False)

print("Baseline output saved successfully!")


Baseline output saved successfully!


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**


##### END-OF-NOTEBOOK CHECKLIST

**IMPORTANT: All items have been verified before proceeding to Notebook 4.**

- [x] Base model (Qwen/Qwen2.5-1.5B-Instruct) loaded in 4-bit quantization without errors
- [x] Baseline output generated for the delivery-period test query
- [x] Baseline response evaluated against the ground-truth SOP rule
- [x] Unsupported statements and hallucination risks documented
- [x] `outputs.json` saved with `test_query`, `ground_truth` and `baseline_output`
- [x] GPU-based model inference executed successfully

**Notebook 3: Baseline Model Evaluation is complete.**


## Overall Summary

In this notebook, the base Qwen2.5-1.5B-Instruct model was loaded using 4-bit quantization to reduce memory requirements. The model was tested using a short customer query about delivery periods ("i need information about delivery periods"), without access to the SOP documents and without any fine-tuning.

The model gave a general explanation of delivery periods and quoted delivery times from its own memory, such as "3-5 days" in the United States. It did not give the delivery timeline specified in the SOP, and the answer was cut off at the token limit.

The response was evaluated against the ground-truth rule that domestic orders deliver within 3–7 business days. The invented delivery times and the missing rule were documented.

The test query, ground-truth rule and generated baseline response were saved in `outputs.json` for use in subsequent notebooks.